In [2]:
!pip3 install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 11.1 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [3]:
import pandas as pd

In [4]:
df = pd.read_json("hf://datasets/spikecodes/911-call-transcripts/calls.jsonl", lines=True)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df

,messages
0,"[{'role': 'assistant', 'content': '9-1-1, what..."
1,"[{'role': 'assistant', 'content': '9-1-1, what..."
2,"[{'role': 'assistant', 'content': '9-1-1, what..."
3,"[{'role': 'assistant', 'content': '9-1-1, what..."
4,"[{'role': 'assistant', 'content': '9-1-1, what..."
...,...
513,"[{'role': 'assistant', 'content': '9-1-1, what..."
514,"[{'role': 'assistant', 'content': '9-1-1, what..."
515,"[{'role': 'assistant', 'content': '9-1-1, what..."
516,"[{'role': 'assistant', 'content': '9-1-1, what..."


In [7]:
df.to_json('911_calls.jsonl', orient='records', lines=True)

In [8]:
# Display sample of data
print(f"Total records: {len(df)}")
print("\nSample call:")
print(df.iloc[0] if len(df) > 0 else "No data")

Total records: 518

Sample call:
messages    [{'role': 'assistant', 'content': '9-1-1, what...
Name: 0, dtype: object


In [11]:
!pip3 install transformers torch accelerate peft datasets --no-build-isolation

  Using cached datasets-4.6.1-py3-none-any.whl.metadata (19 kB)
  Using cached pyarrow-23.0.1-cp314-cp314-macosx_12_0_arm64.whl.metadata (3.1 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached xxhash-3.6.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.18-py313-none-any.whl.metadata (7.2 kB)
  Using cached aiohttp-3.13.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached frozenlist-1.8.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached multidict-6.7.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.3 kB)
  Using cached propcache-0.4.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (13 kB)
  Using cached yarl-1.22.0-cp314-cp314-macosx

In [12]:
# Prepare training data for instruction fine-tuning
# Format: User question -> 911 operator response

def format_training_data(df):
    """Convert call transcripts into instruction-response pairs"""
    training_data = []
    
    for idx, row in df.iterrows():
        # Extract relevant fields from the call transcript
        transcript = row.get('transcript', '')
        
        if not transcript or len(transcript) < 50:
            continue
        
        # Create instruction-response pairs
        # The model will learn to respond as a 911 operator
        instruction = f"""You are an experienced 911 dispatcher. Respond professionally and calmly to the following caller:

{transcript[:500]}"""  # Use first 500 chars as context
        
        response = """Understood. I'm here to help. Let me get some information:
- What is the nature of your emergency?
- What is your current location?
- Is anyone injured?

I'm dispatching help to your location now."""
        
        training_data.append({
            "instruction": instruction,
            "input": "",
            "output": response
        })
    
    return training_data

training_data = format_training_data(df)
print(f"Formatted {len(training_data)} training examples")
if training_data:
    print("\nExample:")
    print(f"Instruction: {training_data[0]['instruction'][:200]}...")
    print(f"Output: {training_data[0]['output']}")

Formatted 0 training examples


In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
from datasets import Dataset

# Save training data
with open('911_operator_training.json', 'w') as f:
    json.dump(training_data, f, indent=2)

# Create Hugging Face dataset
dataset = Dataset.from_dict({
    "instruction": [d["instruction"] for d in training_data],
    "input": [d["input"] for d in training_data],
    "output": [d["output"] for d in training_data]
})

print(f"Dataset created with {len(dataset)} examples")

Dataset created with 0 examples


In [ ]:
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# Load model and tokenizer
model_name = "mistralai/Mistral-7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {model_name}")
print(f"Model size: {model.num_parameters() / 1e9:.2f}B parameters")

In [ ]:
# Apply LoRA for efficient fine-tuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print(f"LoRA applied. Trainable parameters: {model.get_n_trainable_parameters() / 1e6:.2f}M")

In [ ]:
def formatting_func(examples):
    """Format data for training"""
    texts = []
    for instruction, input_text, output in zip(examples['instruction'], examples['input'], examples['output']):
        text = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output}"""
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)

# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text", "instruction", "input", "output"])
tokenized_dataset.set_format("torch")

print("Dataset tokenized and ready for training")

In [ ]:
# Fine-tuning setup
training_args = TrainingArguments(
    output_dir="./mistral-911-operator",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    warmup_steps=100,
    weight_decay=0.01,
    optim="paged_adamw_32bit",
    # Uncomment to save to GPU
    # fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=lambda batch: {k: v.to(model.device) if hasattr(v, 'to') else v for k, v in zip(batch[0].keys(), zip(*[b.values() for b in batch]))},
)

print("Trainer initialized. Ready to start fine-tuning!")
print(f"Training on {len(tokenized_dataset)} examples for {training_args.num_train_epochs} epochs")

In [ ]:
# Start fine-tuning
trainer.train()

print("Fine-tuning complete!")

In [ ]:
from transformers import pipeline

# Test the fine-tuned model
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Test prompts
test_prompts = [
    "I've been in a car accident!",
    "There's a fire in my building!",
    "I think someone is trying to break into my house",
]

print("Testing 911 Operator Model:\n")
for prompt in test_prompts:
    print(f"Caller: {prompt}")
    response = pipe(f"You are a 911 dispatcher. Caller says: {prompt}\n911 Operator: ", 
                    max_length=150, 
                    num_return_sequences=1,
                    temperature=0.7)
    print(f"Response: {response[0]['generated_text'].split('911 Operator: ')[-1]}")
    print("-" * 50)

In [ ]:
# Save the fine-tuned model
model.save_pretrained("./mistral-911-operator-final")
tokenizer.save_pretrained("./mistral-911-operator-final")

print("Model saved to ./mistral-911-operator-final")